In [1]:
import requests
import pandas as pd
import pickle
import numpy as np

In [3]:
raw_data_path = 'data/2025-02-28'
with open(f'{raw_data_path}/OGD_metadata.pkl', 'rb') as file: 
    df = pickle.load(file)
    df.loc[:,'Batch_OGD_date'] = pd.to_datetime(df['Batch_OGD_date'].copy())
    df.sort_values(by='Batch_OGD_date',inplace=True)


In [ ]:
split = 0.8
split_idx  = round(df.shape[0]*split)
df_train = df.iloc[:split_idx,:]
df_val = df.iloc[split_idx:,:]

In [ ]:

with open(f'{raw_data_path}/Analyse_raman.pkl', 'rb') as file: 
    data_raman = pickle.load(file) 
with open(f'{raw_data_path}/Analyse_uv.pkl', 'rb') as file: 
    data_uv = pickle.load(file) 

idx_train = df_train['Batch_OGD_name']
idx_val = df_val['Batch_OGD_name']

In [ ]:

data_raman_train = data_raman[data_raman.index.isin(idx_train)].sort_index()
data_raman_val = data_raman[data_raman.index.isin(idx_val)].sort_index()

data_uv_train = data_uv[data_uv.index.isin(idx_train)].sort_index()
data_uv_val = data_uv[data_uv.index.isin(idx_val)].sort_index()


In [ ]:
from sklearn.decomposition import PCA

reduce_r = PCA(n_components=10)
data_raman_train_weights = reduce_r.fit_transform(data_raman_train)
data_raman_val_weights = reduce_r.transform(data_raman_val)
reduce_uv = PCA(n_components=10)
data_uv_train_weights = reduce_uv.fit_transform(data_uv_train)
data_uv_val_weights = reduce_uv.transform(data_uv_val)

In [ ]:
out_train = pd.DataFrame(np.concatenate([data_uv_train_weights,data_raman_train_weights],axis=1),index=data_raman_train.index)
out_val = pd.DataFrame(np.concatenate([data_uv_val_weights,data_raman_val_weights],axis=1),index=data_raman_val.index)  
